### Install required libraries

In [ ]:
!pip install googletrans==4.0.0-rc1

In [ ]:
!pip install numpy scipy matplotlib googletrans==4.0.0-rc1 faster-whisper silero-vad librosa demucs

### Folder Names Translation into English

In [ ]:
import os
import shutil
from pathlib import Path
from googletrans import Translator
import unicodedata
import string

translator = Translator()
ROOT_DIR = "EchoNet_Data"    
OUTPUT_DIR = "EchoNet_Dataset" 

def is_english(s):
    """Check if the string contains only English (basic ASCII) characters"""
    try:
        return all(c in string.printable for c in s)
    except:
        return False

def translate_text(text, dest_lang='en'):
    """Translate string using Google Translate API"""
    try:
        if is_english(text):
            return text.replace(" ", "_")  
        result = translator.translate(text, dest=dest_lang)
        return result.text.strip().replace(" ", "_")
    except Exception as e:
        print(f"[⚠️] Translation failed for '{text}': {e}")
        return text.replace(" ", "_") 

def translate_path_structure(src_path, dst_root):
    """Recursively translate and copy directory structure and files"""
    for root, dirs, files in os.walk(src_path):
        rel_path = os.path.relpath(root, src_path)
        translated_parts = [translate_text(part) for part in Path(rel_path).parts]
        translated_dir_path = os.path.join(dst_root, *translated_parts)
        os.makedirs(translated_dir_path, exist_ok=True)
        for file in files:
            original_file_path = os.path.join(root, file)
            file_stem, file_ext = os.path.splitext(file)
            translated_file_name = translate_text(file_stem) + file_ext
            new_file_path = os.path.join(translated_dir_path, translated_file_name)
            print(f"Copying: '{original_file_path}' ➝ '{new_file_path}'")
            shutil.copy2(original_file_path, new_file_path)

translate_path_structure(ROOT_DIR, OUTPUT_DIR)

### Lets Start Transcribing.....

In [ ]:
import os
import subprocess
import shutil
import uuid
from pathlib import Path
import numpy as np
import librosa
import soundfile as sf
from scipy.signal import butter, lfilter
import torch
from faster_whisper import WhisperModel
from googletrans import Translator
import json
import matplotlib.pyplot as plt

# === CONFIG ===
OUTPUT_DIR = "output_audio_processing"
LANGUAGE = "en"  # Target language for translation
WHISPER_MODEL_SIZE = "medium"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# === Load Silero VAD ===
torch.set_num_threads(1)
model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad', model='silero_vad')
(get_speech_timestamps, _, read_audio, _, _) = utils

# === 1. Utility Functions ===
def is_audio(file_path):
    return file_path.suffix.lower() in ['.wav', '.mp3', '.aac', '.flac', '.ogg', '.m4a']

def is_video(file_path):
    return file_path.suffix.lower() in ['.mp4', '.mkv', '.avi', '.mov']

def extract_audio_ffmpeg(input_path, output_path):
    command = [
        "ffmpeg", "-y", "-i", str(input_path),
        "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", str(output_path)
    ]
    subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    return butter(order, [low, high], btype='band')

def bandpass_filter(data, lowcut=300, highcut=3400, fs=16000, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order)
    return lfilter(b, a, data)

def normalize_audio(y):
    return y / np.max(np.abs(y))

def plot_fft(data, sample_rate, title="FFT of Audio"):
    freqs = np.fft.rfftfreq(len(data), 1 / sample_rate)
    fft_vals = np.fft.rfft(data)
    plt.figure(figsize=(12, 4))
    plt.plot(freqs, np.abs(fft_vals))
    plt.title(title)
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Magnitude")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(Path(OUTPUT_DIR) / f"{uuid.uuid4().hex}_fft_plot.png")
    plt.close()

def run_demucs(input_audio):
    subprocess.run([
        "python3", "-m", "demucs.separate",
        "--two-stems=vocals", str(input_audio),
        "--out=demucs_out"
    ], check=True)
    base_name = Path(input_audio).stem
    denoised_path = Path(f"demucs_out/htdemucs/{base_name}/vocals.wav")
    return denoised_path

def run_vad_and_split(vad_path, output_base):
    wav = read_audio(str(vad_path), sampling_rate=16000)
    timestamps = get_speech_timestamps(wav, model, return_seconds=True)
    speech_clips = []
    y, sr = librosa.load(vad_path, sr=16000)

    for i, ts in enumerate(timestamps):
        start = int(ts['start'] * sr)
        end = int(ts['end'] * sr)
        clip = y[start:end]
        clip_path = output_base / f"speech_clip_{i}.wav"
        sf.write(clip_path, clip, sr)
        speech_clips.append(clip_path)

    return speech_clips

def transcribe_audio(model, audio_path):
    segments, _ = model.transcribe(str(audio_path))
    if segments is None:
        return ""
    transcript = " ".join([s.text.strip() for s in segments])
    return transcript

def detect_and_translate(text):
    translator = Translator()
    if not text.strip():
        return "unknown", ""
    detected = translator.detect(text)
    if detected.lang != LANGUAGE:
        translated = translator.translate(text, dest=LANGUAGE)
        return detected.lang, translated.text
    return detected.lang, text

# === 2. Main Pipeline ===
def process_audio_file(file_path):
    print(f"Processing: {file_path.name}")
    tmp_wav = Path(OUTPUT_DIR) / f"{uuid.uuid4().hex}_converted.wav"

    if is_video(file_path):
        extract_audio_ffmpeg(file_path, tmp_wav)
    elif is_audio(file_path):
        extract_audio_ffmpeg(file_path, tmp_wav)
    else:
        print("❌ Unsupported file format:", file_path)
        return

    y, sr = librosa.load(tmp_wav, sr=16000)
    plot_fft(y, sr, title=f"FFT of Original Audio: {file_path.name}")

    y_filtered = bandpass_filter(y, fs=sr)
    y_normalized = normalize_audio(y_filtered)
    filtered_path = tmp_wav.with_name(tmp_wav.stem + "_filtered.wav")
    sf.write(filtered_path, y_normalized, sr)

    print("🔧 Running Demucs denoiser...")
    denoised_path = run_demucs(filtered_path)

    print("🎤 Running VAD and splitting...")
    vad_segments = run_vad_and_split(denoised_path, tmp_wav.parent)

    print("🧠 Loading Whisper model...")
    compute_type = "float32"  # Safe fallback
    if torch.cuda.is_available():
        compute_type = "float16"
    model = WhisperModel(WHISPER_MODEL_SIZE, compute_type=compute_type)

    results = []
    for seg in vad_segments:
        print(f"🔎 Transcribing {seg.name}")
        transcription = transcribe_audio(model, seg)
        original_lang, translated_text = detect_and_translate(transcription)

        results.append({
            "segment_file": str(seg),
            "original_transcription": transcription,
            "detected_language": original_lang,
            "translated_transcription": translated_text
        })

    # Save to JSON
    output_json = tmp_wav.with_suffix('.json')
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

    print(f"✅ Done: Transcriptions saved to {output_json}\n")

def process_input(input_path):
    input_path = Path(input_path)
    if input_path.is_file():
        process_audio_file(input_path)
    elif input_path.is_dir():
        for f in input_path.iterdir():
            if is_video(f) or is_audio(f):
                process_audio_file(f)
    else:
        print("❌ Invalid input path.")

process_input("/home/fahad/Desktop/EchoNet-Dataset/EchoNet_Dataset")
#process_input("EchoNet_Dataset")